In [1]:
import pandas as pd
from pyspark.sql import functions as f
from pyspark.sql import DataFrame
from pyspark.sql.types import DecimalType, StringType, FloatType
import datetime
from dateutil.relativedelta import relativedelta
import numpy as np
import json
import logging as logger

# transform categorical features
from pyspark.ml import Pipeline
from pyspark.ml.feature import (
    OneHotEncoder, 
    StringIndexer, 
    VectorAssembler
)
from synapse.ml.lightgbm import LightGBMClassifier
import pandas as pd

logger.basicConfig(
    format='%(asctime)s - %(levelname)s - %(message)s',
    level=logger.INFO,
)

pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)

# import sys
# sys.path.append('/Workspace/Repos/xi.chen@gucci.com/dscchn-ec_client_forecasting/')

## Params

In [ ]:
categorical_features = ['2_year_dos_sales','2_year_frequency','3_year_dos_sales','recency','tenure_day','3_year_frequency','frequency_ca_reached','1_year_frequency','all_frequency','recency_ca_reached','all_amt_last_2','all_dos_sales','days_to_birthday_no_jump','frequency_sms','1_year_rtw_qty','higher_ave_times']
numerical_features = ['provincepy','classification_code','age_group']
train_sample_table = "china_data_science_dev.ai_recommendation.cf_train_sample_test"

target = '15_Dos_Purchase'

## Functions

In [ ]:

def preprocess_features(pdf, categorical_features, numerical_features, fillna_dict):
    '''
    Preprocess features for pyspark ml
    - assign string to categorical features
    - fill na
    pdf: pyspark_dataframe to be preprocessed
    categorical_features: list of categorical features
    numerical_features: list of numerical features
    fillna_dict: set in configuration with key-value pairs
    '''
    # assign string type
    for cat_col in categorical_features:
        pdf = pdf.withColumn(cat_col, pdf[cat_col].cast(StringType()))
    # fill na
    if fillna_dict:
        pdf = pdf.fillna(fillna_dict).fillna(0, subset = numerical_features)

    return pdf

In [ ]:
def lightgbm_classification_training(train_pdf,
                                    hyperparameter_dictionary,
                                    categorical_features, 
                                    numerical_features,
                                    mlflow):

    # Create a StringIndexer for each categorical column
    indexers     = [StringIndexer(inputCol = col, outputCol=col + "_index", handleInvalid="keep") for col in categorical_features]
    feature_cols = numerical_features + [col + "_index" for col in categorical_features]
    assembler    = VectorAssembler(inputCols = feature_cols, outputCol = "features")
    model  = LightGBMClassifier(**hyperparameter_dictionary)
    # train_data = assembler.transform(train_pdf)
    stages = [index for index in indexers] + [assembler, model]
    pipeline = Pipeline(stages = stages)
    pipelineModel = pipeline.fit(train_pdf)
    model = pipelineModel.stages[-1]

    ## get feature importance
    feature_importances = model.getFeatureImportances()
    fi = pd.Series(feature_importances, index=feature_cols)
    fi = fi.sort_values(ascending=False)
    fi = fi.to_dict()

    if mlflow:
        # log model
        mlflow.spark.log_model(pipelineModel, 'model')
        # log feature importance
        mlflow.log_dict(fi, 'feature_importance.json')

    return pipelineModel, fi

## Prep Training Samples

In [ ]:
# train_sample = spark.sql(f'''
#     SELECT 
#         sha2(customer_code, 256) AS customer_code,
#         provincepy,
#         2_year_dos_sales,
#         2_year_frequency,
#         classification_code,
#         3_year_dos_sales,
#         recency,
#         tenure_day,
#         3_year_frequency,
#         frequency_ca_reached,
#         age_group,
#         1_year_frequency,
#         all_frequency,
#         recency_ca_reached,
#         all_amt_last_2,
#         all_dos_sales,
#         days_to_birthday_no_jump,
#         frequency_sms,
#         1_year_rtw_qty,
#         higher_ave_times,
#         `15_Dos_purchase`
#     FROM china_data_science_prod.ai_recommendation.dos_resampling_data_ga4
#     LIMIT 100
# ''')

# train_sample.write.mode("overwrite").saveAsTable("china_data_science_dev.ai_recommendation.cf_train_sample_test")
# display(train_sample)

## Training Pipeline

In [ ]:
train_samp = spark.sql(f'''
                       SELECT * FROM {train_sample_table}
                       ''')
train_samp = train_samp.select(categorical_features + numerical_features+[target])

In [ ]:
MODEL_PARAMS = {
                'objective':"binary",
                'featuresCol':"features", 
                'labelCol': "15_Dos_purchase",
                'isUnbalance':True,
                'learningRate':0.05,
                'numIterations':200, 
                'numLeaves':31, 
                'maxDepth':7
         } 
EXPERIMENT_ID = "1234567"
cutoff_date = "9999-99-99"

In [ ]:
# preprocess features 
train_samp = preprocess_features(pdf = train_samp, 
                                 categorical_features = categorical_features, 
                                 numerical_features = numerical_features, 
                                 fillna_dict = None)
train_samp_columns = train_samp.columns
logger.info(f'tran_samp_columns: {train_samp_columns}')

# train
# # # Set the registry URI to Databricks Unity Catalog
# mlflow.set_registry_uri("databricks-uc")
# run = mlflow.start_run(experiment_id = EXPERIMENT_ID, run_name = cutoff_date)

model, feature_importance = lightgbm_classification_training(train_pdf = train_samp,
                                                             hyperparameter_dictionary = MODEL_PARAMS,
                                                             categorical_features = categorical_features, 
                                                             numerical_features = numerical_features,
                                                             mlflow = None)